# Skin Cancer Detection — 2ᵉ backbone (ConvNeXt-Tiny) + ensemble
Objectif : améliorer l'accuracy et la stabilité en ajoutant un modèle **d'architecture différente**
au modèle EfficientNetV2-S + CBAM (notebook `skincanerf-phase3`), puis en les combinant.

- Mêmes splits (relus depuis les CSV), même pipeline, même calibration → comparaison équitable.
- Les prédictions TTA d'EfficientNetV2-S sont **réutilisées** (pas de recalcul).
- Reprise automatique sur NaN + **garde-fou de temps** (arrêt propre avant la limite de 12 h de Kaggle).
- Le choix final (EfficientNet seul, ConvNeXt seul ou ensemble) est décidé **sur VAL** ; le TEST n'est utilisé qu'une fois.

**Inputs requis** : Output du notebook `skincanerf-phase3` + datasets ISIC2019 (cdeotte), HAM10000 (surajghuwalewala), PH2 (spacesurfer).
**Accelerator** : GPU T4 x2. **Internet** : activé (téléchargement des poids ImageNet de ConvNeXt).
**Durée estimée** : ~5 à 7 h.

## Step 1 — Setup

In [1]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

mixed_precision.set_global_policy("mixed_float16")

# ─── Constantes globales ───────────────────────────────────
IMG_SIZE   = 384
BATCH_SIZE = 32      
EVAL_RESIZE  = int(IMG_SIZE * 1.15)   # resize avant center-crop à l'inférence
NUM_CLASSES  = 7
AUTOTUNE     = tf.data.AUTOTUNE

# Régularisation
LABEL_SMOOTH = 0.1
WEIGHT_DECAY = 1e-4
MIXUP_ALPHA  = 0.1

# Mapping stable (utilisé partout dans le projet)
CLASSES      = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_NAMES_FULL = {
    "akiec": "Actinic keratoses",
    "bcc":   "Basal cell carcinoma",
    "bkl":   "Benign keratosis-like lesions",
    "df":    "Dermatofibroma",
    "nv":    "Melanocytic nevi",
    "mel":   "Melanoma",
    "vasc":  "Vascular lesions",
}

print("TF:", tf.__version__)
print("GPUs:", gpus)
print("Mixed precision:", mixed_precision.global_policy())
print("Mapping:", CLASS_TO_IDX)


import glob, time
T0       = time.time()
BUDGET_H = 9.5          # arrêt propre de l'entraînement après 9 h 30 (Kaggle coupe à 12 h)

assert len(gpus) > 0, "❌ PAS DE GPU → Settings → Accelerator → GPU T4 x2"
if len(gpus) < 2:
    print("⚠️  1 seul GPU détecté : ça marche, mais ~2x plus lent (choisis GPU T4 x2).")

# ─── Localiser l'Output du notebook skincanerf-phase3 ───
INPUT_ROOT = "/kaggle/input"
cands = sorted(set(sum([glob.glob(f"{INPUT_ROOT}/{'*/' * d}effnetv2s_cbam_final.keras")
                        for d in range(1, 4)], [])))
assert cands, ("❌ Output de skincanerf-phase3 introuvable → "
               "+ Add Input → onglet Notebooks → skincanerf-phase3")
PHASE3_DIR = os.path.dirname(cands[0])
print("PHASE3_DIR :", PHASE3_DIR)
OUT_DIR = "/kaggle/working"

strategy = tf.distribute.MirroredStrategy(
    cross_device_ops=tf.distribute.ReductionToOneDevice())
print("Répliques:", strategy.num_replicas_in_sync)   # doit afficher 2

TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision: <DTypePolicy "mixed_float16">
Mapping: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'nv': 4, 'mel': 5, 'vasc': 6}
PHASE3_DIR : /kaggle/input/notebooks/rihembousbih/skincanerf-phase3
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Répliques: 2


I0000 00:00:1790159606.110812      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790159606.114011      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


## Step 2 — Pipeline tf.data, callbacks, loss (identiques au run EfficientNet)

In [2]:
class NanWatch(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        if batch % 50:
            return
        for v in self.model.variables:
            if not bool(tf.reduce_all(tf.math.is_finite(tf.cast(v, tf.float32)))):
                print(f"\n⚠️  NON-FINI dans « {v.path} » au batch {batch}")
                self.model.stop_training = True
                return

In [3]:
PREPROCESS = lambda x: x        # ConvNeXt : preprocessing intégré au modèle (entrée 0-255)

def decode_image(path):
    """Décode jpg/png/bmp selon l'extension."""
    img_bytes = tf.io.read_file(path)
    ext = tf.strings.lower(tf.strings.split(path, ".")[-1])

    def _jpg(): return tf.image.decode_jpeg(img_bytes, channels=3)
    def _png(): return tf.image.decode_png(img_bytes,  channels=3)
    def _bmp(): return tf.image.decode_bmp(img_bytes)

    img = tf.case(
        [(tf.equal(ext, "jpg"),  _jpg),
         (tf.equal(ext, "jpeg"), _jpg),
         (tf.equal(ext, "png"),  _png),
         (tf.equal(ext, "bmp"),  _bmp)],
        default=_jpg, exclusive=True
    )
    return tf.ensure_shape(img, [None, None, 3])


# ── NOUVEAU : normalisation d'illuminant (Shades of Gray, p=6) ──
# Neutralise la balance des blancs du dermatoscope. C'est le correctif
# principal pour l'écart de généralisation observé sur PH2.
def shades_of_gray(img, p=6.0):
    """img : float32 en échelle 0-255. Retourne float32 0-255."""
    flat  = tf.reshape(img, [-1, 3])
    illum = tf.pow(tf.reduce_mean(tf.pow(flat + 1e-6, p), axis=0), 1.0 / p)
    illum = illum / (tf.norm(illum) + 1e-6)
    img   = img / (illum * tf.sqrt(3.0) + 1e-6)
    return tf.clip_by_value(img, 0.0, 255.0)


def augment_train(img):
    """img : float32, 0-255, AVANT resnet_preprocess."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))

    # MODIFIÉ : amplitudes doublées pour couvrir la variabilité inter-appareils
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.30)      # était 0.15
    img = tf.image.random_contrast(img,   0.6, 1.4)  # était 0.8, 1.2
    img = tf.image.random_saturation(img, 0.6, 1.4)  # était 0.8, 1.2
    img = tf.image.random_hue(img,        0.08)      # était 0.03
    img = tf.clip_by_value(img, 0.0, 1.0) * 255.0

    # cutout à forme statique : masque booléen sur une grille fixe
    h  = tf.random.uniform([], IMG_SIZE//8, IMG_SIZE//4, dtype=tf.int32)
    y0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    yy = tf.range(IMG_SIZE)[:, None]
    xx = tf.range(IMG_SIZE)[None, :]
    inside = (yy >= y0) & (yy < y0 + h) & (xx >= x0) & (xx < x0 + h)
    apply_cut = tf.cast(tf.random.uniform([]) < 0.5, tf.float32)
    keep = 1.0 - apply_cut * tf.cast(inside, tf.float32)[:, :, None]
    img = img * keep
    return tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])


def load_and_preprocess(path, label, training=False):
    img = tf.cast(decode_image(path), tf.float32)

    if training:
        shape = tf.shape(img)
        scale = tf.random.uniform([], 0.7, 1.0)
        h = tf.cast(tf.cast(shape[0], tf.float32) * scale, tf.int32)
        w = tf.cast(tf.cast(shape[1], tf.float32) * scale, tf.int32)
        img = tf.image.random_crop(img, [h, w, 3])
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = augment_train(img)
    else:
        # MODIFIÉ : resize + center-crop, au lieu d'un resize direct.
        # L'entraînement voit des crops à 70-100 % ; un resize plein cadre
        # à l'inférence crée un décalage d'échelle systématique.
        img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
        off = (EVAL_RESIZE - IMG_SIZE) // 2
        img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)

    img   = shades_of_gray(img)          # NOUVEAU — train ET inférence
    img   = PREPROCESS(img)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label


def make_dataset(df, training=False, cache=False, shuffle_buffer=4096, batched=True):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["path"].values.astype(str), df["label"].values.astype(np.int32))
    )
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_and_preprocess(p, y, training=training),
                num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if batched:                              # NOUVEAU : option non-batchée
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


# ── NOUVEAU : mixup ──
def mixup(ds, alpha=MIXUP_ALPHA):
    """À appliquer APRÈS .batch(). Produit des labels mous."""
    def _mix(imgs, labels):
        b   = tf.shape(imgs)[0]
        g1  = tf.random.gamma([b], alpha)
        g2  = tf.random.gamma([b], alpha)
        lam = g1 / (g1 + g2)                  # Beta(alpha, alpha)
        idx = tf.random.shuffle(tf.range(b))
        li  = tf.reshape(lam, [b, 1, 1, 1])
        ll  = tf.reshape(lam, [b, 1])
        return (li * imgs   + (1 - li) * tf.gather(imgs,   idx),
                ll * labels + (1 - ll) * tf.gather(labels, idx))
    return ds.map(_mix, num_parallel_calls=AUTOTUNE)


# ── NOUVEAU : échantillonnage équilibré (remplace la duplication) ──
def balanced_dataset(df, power=0.5):
    """Poids par classe ∝ n^power. power=0.5 (racine) = compromis usuel ;
    power=0 = uniforme strict ; power=1 = distribution naturelle."""
    dss, weights = [], []
    for c in CLASSES:
        sub = df[df["dx"] == c]
        if len(sub) == 0:
            continue
        d = make_dataset(sub, training=True,
                         shuffle_buffer=min(len(sub), 4096), batched=False)
        dss.append(d.repeat())
        weights.append(float(len(sub)) ** power)
    w = np.array(weights) / np.sum(weights)
    ds = tf.data.Dataset.sample_from_datasets(dss, weights=list(w), seed=SEED)
    opts = tf.data.Options()
    opts.experimental_distribute.auto_shard_policy = \
        tf.data.experimental.AutoShardPolicy.DATA
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE).with_options(opts)


print("Pipeline tf.data défini (shades-of-gray + mixup + balanced sampling).")

Pipeline tf.data défini (shades-of-gray + mixup + balanced sampling).


In [4]:
from sklearn.metrics import f1_score
from tensorflow.keras.optimizers.schedules import CosineDecay


class MacroF1(tf.keras.callbacks.Callback):
    """Calcule val_macro_f1 en fin d'époque. À placer EN PREMIER dans callbacks."""
    def __init__(self, val_ds, y_true):
        super().__init__()
        self.ds, self.y = val_ds, y_true

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = np.argmax(self.model.predict(self.ds, verbose=0), axis=1)
        logs["val_macro_f1"] = f1_score(self.y, p, average="macro")
        print(f"   val_macro_f1: {logs['val_macro_f1']:.4f}")


def make_callbacks(val_ds, y_val, ckpt_path, patience=10, best=None):
    """best : score à battre pour écraser le checkpoint (utile en cas de reprise)."""
    return [
        MacroF1(val_ds, y_val),
        NanWatch(),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_macro_f1",
                                           mode="max", save_best_only=True,
                                           initial_value_threshold=best),
        tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                         patience=patience,
                                         restore_best_weights=True),
    ]


def cosine_lr(peak_lr, n_epochs, n_train, warmup_epochs=2):
    spe = max(n_train // BATCH_SIZE, 1)
    return CosineDecay(initial_learning_rate=peak_lr / 10,
                       decay_steps=spe * n_epochs,
                       warmup_target=peak_lr,
                       warmup_steps=spe * warmup_epochs,
                       alpha=0.01)


print("Callbacks et schedule définis.")
def make_optimizer(peak_lr, n_epochs, n_train, warmup_epochs=2):
    return tf.keras.optimizers.AdamW(
        learning_rate=cosine_lr(peak_lr, n_epochs, n_train, warmup_epochs),
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0,
    )

CE_SMOOTH = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)

Callbacks et schedule définis.


## Step 3 — Splits d'origine (identiques à ceux d'EfficientNet)

In [5]:
import pandas as pd

train_df = pd.read_csv(f"{PHASE3_DIR}/split_train.csv")
val_df   = pd.read_csv(f"{PHASE3_DIR}/split_val.csv")
test_df  = pd.read_csv(f"{PHASE3_DIR}/split_test.csv")
ph2_df   = pd.read_csv(f"{PHASE3_DIR}/split_ph2.csv")

# ─── Les images sont-elles accessibles ? ───
for nom, df in [("train", train_df), ("val", val_df), ("test", test_df), ("ph2", ph2_df)]:
    manquants = (~df["path"].map(os.path.exists)).sum()
    print(f"{nom:5s}: {len(df):6d} images, {manquants} introuvables")
    assert manquants == 0, (f"❌ Images {nom} introuvables (ex: {df['path'].iloc[0]}) → "
                            "ajoute en Input les datasets ISIC2019 / HAM10000 / PH2")

# ─── Anti-fuite ───
for a, b, nom in [(train_df, val_df, "train/val"), (train_df, test_df, "train/test"),
                  (val_df, test_df, "val/test")]:
    n = len(set(a["lesion_id"]) & set(b["lesion_id"]))
    assert n == 0, f"❌ {n} lésions communes {nom}"
print("✅ 0 lésion commune entre train / val / test")

val_ds = make_dataset(val_df, training=False)
y_val, y_test, y_ph2 = (val_df["label"].values, test_df["label"].values, ph2_df["label"].values)
print("\nDistribution train :", train_df["dx"].value_counts().to_dict())

train:  17455 images, 0 introuvables
val  :   3707 images, 0 introuvables
test :   3738 images, 0 introuvables
ph2  :    197 images, 0 introuvables
✅ 0 lésion commune entre train / val / test

Distribution train : {'nv': 9005, 'mel': 3151, 'bcc': 2369, 'bkl': 1833, 'akiec': 753, 'df': 175, 'vasc': 169}


## Step 4 — Modèle ConvNeXt-Tiny + boucle d'entraînement robuste

In [6]:
import shutil
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ConvNeXtTiny


def build_convnext():
    base = ConvNeXtTiny(include_top=False, weights="imagenet",
                        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    x   = layers.GlobalAveragePooling2D(name="gap", dtype="float32")(base.output)
    x   = layers.LayerNormalization(name="head_ln", dtype="float32")(x)
    x   = layers.Dropout(0.3, name="head_drop", dtype="float32")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="pred", dtype="float32")(x)
    return Model(base.input, out, name="convnext_tiny_skin"), base


class TimeBudget(tf.keras.callbacks.Callback):
    """Arrête proprement l'entraînement si le budget temps est dépassé."""
    def on_epoch_end(self, epoch, logs=None):
        h = (time.time() - T0) / 3600
        if h > BUDGET_H:
            print(f"\n⏱️  Budget temps atteint ({h:.1f} h) → arrêt propre, meilleur checkpoint conservé.")
            self.model.stop_training = True


def train_with_restarts(model, train_ds, steps, epochs, peak_lr, ckpt, start_ckpt=None,
                        best=None, patience=6, warmup=1, max_restarts=3):
    """Entraîne avec reprise automatique depuis le meilleur checkpoint en cas de NaN."""
    if start_ckpt is not None:
        shutil.copy(start_ckpt, ckpt)            # le checkpoint n'est écrasé que si on fait mieux
    best = -np.inf if best is None else best
    epochs_done, lr = 0, peak_lr
    for attempt in range(max_restarts + 1):
        remaining = epochs - epochs_done
        if remaining <= 0 or (time.time() - T0) / 3600 > BUDGET_H:
            break
        print(f"\n▶ Tentative {attempt + 1} : {remaining} époques, lr max = {lr:.1e}, "
              f"F1 à battre = {best:.4f}")
        with strategy.scope():
            model.compile(optimizer=make_optimizer(lr, remaining, len(train_df),
                                                   warmup_epochs=warmup if attempt == 0 else 1),
                          loss=CE_SMOOTH, metrics=["accuracy"])
        cbs = make_callbacks(val_ds, y_val, ckpt, patience=patience,
                             best=best if np.isfinite(best) else None) + [TimeBudget()]
        h = model.fit(train_ds, steps_per_epoch=steps, validation_data=val_ds,
                      epochs=remaining, callbacks=cbs, verbose=1)
        epochs_done += len(h.history["loss"])
        best = max([best] + [v for v in h.history.get("val_macro_f1", []) if np.isfinite(v)])

        poids_ok = all(np.isfinite(w).all() for w in model.get_weights())
        if np.isfinite(h.history["loss"][-1]) and poids_ok:
            print(f"\n✅ Terminé ({epochs_done} époques, {(time.time() - T0) / 3600:.1f} h écoulées).")
            break
        print(f"\n⚠️  NaN après {epochs_done} époques → rechargement du meilleur checkpoint, lr / 2")
        if not os.path.exists(ckpt):
            raise RuntimeError("NaN avant le premier checkpoint : relance le notebook.")
        model.load_weights(ckpt)
        lr /= 2
    else:
        print("\n⚠️  Nombre max de reprises atteint — on garde le meilleur checkpoint.")
    return best, epochs_done


with strategy.scope():
    model, base = build_convnext()
print(f"ConvNeXt-Tiny construit : {model.count_params() / 1e6:.1f} M paramètres")

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
ConvNeXt-Tiny construit : 27.8 M paramètres


## Step 5 — Phase 1 : backbone gelé, entraînement de la tête (~30-45 min)

In [7]:
CKPT_P1    = f"{OUT_DIR}/convnext_tiny_phase1.keras"
EPOCHS_P1  = 6

base.trainable = False
train_ds = make_dataset(train_df, training=True)
best_p1, ep_p1 = train_with_restarts(model, train_ds, None, EPOCHS_P1, peak_lr=1e-3,
                                     ckpt=CKPT_P1, patience=3, warmup=1)
print(f"Phase 1 : meilleur val_macro_f1 = {best_p1:.4f}")


▶ Tentative 1 : 6 époques, lr max = 1.0e-03, F1 à battre = -inf
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job

I0000 00:00:1790159710.823539      69 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step - accuracy: 0.4142 - loss: 1.9268INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
   val_macro_f1: 0.3042
546/546 ━━━━━━━━━━━━━━━━━━━━ 434s 695ms/step - accuracy: 0.5040 - loss: 1.5749 - val_accuracy: 0.6258 - val_loss: 1.2446 - val_macro_f1: 0.3042
Epoch 2/6
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step - accuracy: 0.6731 - loss: 1.1556   val_macro_f1: 0.3510
546/546 ━━━━━━━━━━━━━━━━━━━━ 206s 376ms/step - accuracy: 0.6254 - loss: 1.2406 - val_accuracy: 0.6601 - val_loss: 1.1816 - val_macro_f1: 0.3510
Epoch 3/6
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 0.6900 - loss: 1.1039   val_macro_f1: 0.3598
546/546 ━━━━━━━━━━━━━━━━━━━━ 203s 371ms/step - accuracy: 0.6356 - loss: 1.2060 - val_accuracy: 0.6623 - val_loss

## Step 6 — Phase 2 : fine-tuning complet (mixup + échantillonnage équilibré)
Le temps par époque s'affiche : si c'est trop lent, le garde-fou arrête proprement à 9 h 30.

In [8]:
CKPT_FINAL = f"{OUT_DIR}/convnext_tiny_final.keras"
EPOCHS_P2  = 25

model.load_weights(CKPT_P1)
base.trainable = True
train_ds_bal = mixup(balanced_dataset(train_df, power=0.5))
STEPS = len(train_df) // BATCH_SIZE

best_p2, ep_p2 = train_with_restarts(model, train_ds_bal, STEPS, EPOCHS_P2, peak_lr=4e-5,
                                     ckpt=CKPT_FINAL, start_ckpt=CKPT_P1, best=best_p1,
                                     patience=6, warmup=2)
print(f"Phase 2 : meilleur val_macro_f1 = {best_p2:.4f}  (phase 1 : {best_p1:.4f})")


▶ Tentative 1 : 25 époques, lr max = 4.0e-05, F1 à battre = 0.3693
Epoch 1/25
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/de

2026-09-23 11:00:50.571793: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 11:00:50.695045: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 11:00:50.733710: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 11:00:50.848789: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 11:00:50.873291: E external/local_xla/xla/stream_

545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 791ms/step - accuracy: 0.5695 - loss: 1.3790   val_macro_f1: 0.5536
545/545 ━━━━━━━━━━━━━━━━━━━━ 773s 1s/step - accuracy: 0.5705 - loss: 1.3699 - val_accuracy: 0.7079 - val_loss: 1.1008 - val_macro_f1: 0.5536
Epoch 2/25
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 799ms/step - accuracy: 0.5912 - loss: 1.3200   val_macro_f1: 0.6022
545/545 ━━━━━━━━━━━━━━━━━━━━ 494s 906ms/step - accuracy: 0.6252 - loss: 1.2663 - val_accuracy: 0.7022 - val_loss: 1.0906 - val_macro_f1: 0.6022
Epoch 3/25
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 809ms/step - accuracy: 0.6766 - loss: 1.1723   val_macro_f1: 0.6667
545/545 ━━━━━━━━━━━━━━━━━━━━ 499s 916ms/step - accuracy: 0.6761 - loss: 1.1682 - val_accuracy: 0.7675 - val_loss: 0.9354 - val_macro_f1: 0.6667
Epoch 4/25
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 806ms/step - accuracy: 0.7118 - loss: 1.1064   val_macro_f1: 0.6952
545/545 ━━━━━━━━━━━━━━━━━━━━ 497s 912ms/step - accuracy: 0.7241 - loss: 1.0778 - val_accuracy: 0.7845 - val_loss: 0.9176 - val_macro_f1: 0

In [9]:
# ─── Contrôle du checkpoint final (sur disque) ───
model_cx = tf.keras.models.load_model(CKPT_FINAL, compile=False)
p = model_cx.predict(val_ds, verbose=0)
assert np.isfinite(p).all(), "❌ NaN dans le checkpoint final"
print(f"✅ Checkpoint ConvNeXt sain. val_macro_f1 (sans TTA) = {f1_score(y_val, p.argmax(1), average='macro'):.4f}")
del model

✅ Checkpoint ConvNeXt sain. val_macro_f1 (sans TTA) = 0.7883


## Step 7 — Fonctions d'évaluation (TTA, calibration, bootstrap)

In [10]:
import time
from scipy.optimize import minimize, differential_evolution
from sklearn.metrics import (accuracy_score, f1_score, recall_score, log_loss, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import label_binarize

MEL = CLASS_TO_IDX["mel"]
CIBLE_SENS_MEL = 0.85


def predict_tta(model, df, n_rot=4):
    """Moyenne sur 4 rotations x 2 flips, prétraitement identique à l'inférence."""
    probs = np.zeros((len(df), NUM_CLASSES), dtype=np.float64)
    for k in range(n_rot):
        for flip in (False, True):
            def _map(p, y, k=k, flip=flip):
                img = tf.cast(decode_image(p), tf.float32)
                img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
                off = (EVAL_RESIZE - IMG_SIZE) // 2
                img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
                img = tf.image.rot90(img, k=k)
                if flip:
                    img = tf.image.flip_left_right(img)
                img = shades_of_gray(img)
                return PREPROCESS(img), y
            ds = (tf.data.Dataset.from_tensor_slices(
                      (df["path"].values.astype(str), df["label"].values.astype(np.int32)))
                  .map(_map, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH_SIZE).prefetch(AUTOTUNE))
            probs += model.predict(ds, verbose=0)
    return probs / (n_rot * 2)


def preds_cached(model, name, df, split):
    f = f"{OUT_DIR}/preds_{split}_{name}.npy"
    if os.path.exists(f):
        return np.load(f)
    t0 = time.time()
    p = predict_tta(model, df)
    np.save(f, p)
    print(f"  {split:4s}/{name:5s} : {time.time() - t0:.0f}s")
    return p


def ensemble_weighted(preds_list, weights):
    w = np.abs(np.array(weights, dtype=float))
    w = w / (w.sum() + 1e-12)
    return sum(w[i] * preds_list[i] for i in range(len(preds_list)))


def apply_temperature(probs, T):
    logits = np.log(np.clip(probs, 1e-12, 1 - 1e-12))
    s = np.exp(logits / T)
    return s / s.sum(axis=1, keepdims=True)


def find_temperature(probs_val, y):
    def nll(x):
        return log_loss(y, apply_temperature(probs_val, float(x[0])),
                        labels=np.arange(probs_val.shape[1]))
    return float(minimize(nll, x0=[1.0], bounds=[(0.05, 5.0)], method="L-BFGS-B").x[0])


def apply_thresholds(probs, thr):
    thr = np.clip(np.asarray(thr, dtype=float), 0.05, 5.0)
    return np.argmax(probs / thr[None, :], axis=1)


def optimize_thresholds(probs_val, y):
    """Maximise l'accuracy VAL sous contrainte sensibilité mélanome >= CIBLE_SENS_MEL."""
    def obj(thr):
        pred = apply_thresholds(probs_val, thr)
        sens = recall_score(y, pred, labels=[MEL], average="macro", zero_division=0)
        if sens < CIBLE_SENS_MEL:
            return 10.0 + (CIBLE_SENS_MEL - sens) * 100.0
        return -accuracy_score(y, pred)
    res = differential_evolution(obj, [(0.2, 3.0)] * probs_val.shape[1],
                                 maxiter=80, popsize=10, polish=False, seed=SEED)
    return np.clip(res.x, 0.2, 3.0)


# ─── Métriques + bootstrap ───
acc    = lambda yt, yp: accuracy_score(yt, yp)
mf1    = lambda yt, yp: f1_score(yt, yp, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0)
s_mel  = lambda yt, yp: ((yt == MEL) & (yp == MEL)).sum() / max((yt == MEL).sum(), 1)
sp_mel = lambda yt, yp: ((yt != MEL) & (yp != MEL)).sum() / max((yt != MEL).sum(), 1)
M7 = {"Accuracy": acc, "Macro-F1": mf1, "Sensibilité mel": s_mel, "Spécificité mel": sp_mel}


def bootstrap_ci(yt, yp, metric, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = [metric(yt[i], yp[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])


def report(nom, yt, yp, metrics=M7):
    print(f"\n=== {nom} ({len(yt)} images) ===")
    out = {}
    for mname, m in metrics.items():
        v = m(yt, yp); lo, hi = bootstrap_ci(yt, yp, m)
        out[mname] = (float(v), float(lo), float(hi))
        print(f"  {mname:18s} {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")
    return out

print("Fonctions d'évaluation définies.")

Fonctions d'évaluation définies.


## Step 8 — Prédictions : ConvNeXt (TTA, ~15 min) + EfficientNet (réutilisées)

In [11]:
P_cx  = {s: preds_cached(model_cx, "convnext", df, s)
         for s, df in [("val", val_df), ("test", test_df), ("ph2", ph2_df)]}
P_eff = {s: np.load(f"{PHASE3_DIR}/preds_{s}_final.npy") for s in ["val", "test", "ph2"]}

for s, n in [("val", len(val_df)), ("test", len(test_df)), ("ph2", len(ph2_df))]:
    assert len(P_eff[s]) == len(P_cx[s]) == n, f"❌ Désalignement sur {s}"
print(f"Contrôle EfficientNet : VAL macro-F1 = {mf1(y_val, P_eff['val'].argmax(1)):.4f} (attendu 0.8010)")

print("\n=== Modèles individuels (TTA, argmax brut) ===")
for name, Pm in [("EfficientNetV2-S", P_eff), ("ConvNeXt-Tiny", P_cx)]:
    pv, pt = Pm["val"].argmax(1), Pm["test"].argmax(1)
    print(f"{name:17s} VAL Acc={acc(y_val, pv):.4f} F1={mf1(y_val, pv):.4f}   "
          f"TEST Acc={acc(y_test, pt):.4f} F1={mf1(y_test, pt):.4f}")

  val /convnext : 269s
  test/convnext : 254s
  ph2 /convnext : 33s
Contrôle EfficientNet : VAL macro-F1 = 0.8010 (attendu 0.8010)

=== Modèles individuels (TTA, argmax brut) ===
EfficientNetV2-S  VAL Acc=0.8632 F1=0.8010   TEST Acc=0.8604 F1=0.7895
ConvNeXt-Tiny     VAL Acc=0.8530 F1=0.7964   TEST Acc=0.8537 F1=0.7971


## Step 9 — Ensemble : poids optimisés sur VAL, choix final décidé sur VAL

In [12]:
NAMES = ["effnetv2s", "convnext"]
PV = [P_eff["val"],  P_cx["val"]]
PT = [P_eff["test"], P_cx["test"]]
PP = [P_eff["ph2"],  P_cx["ph2"]]

res = differential_evolution(lambda w: -mf1(y_val, ensemble_weighted(PV, w).argmax(1)),
                             bounds=[(0, 1)] * 2, seed=SEED, maxiter=40, tol=1e-4, workers=1)
W_ens = np.abs(res.x) / np.abs(res.x).sum()
f1_ens    = -res.fun
f1_moy    = mf1(y_val, ensemble_weighted(PV, [0.5, 0.5]).argmax(1))
f1_single = [mf1(y_val, p.argmax(1)) for p in PV]
i_best    = int(np.argmax(f1_single))

print("Poids optimisés (VAL) :", dict(zip(NAMES, W_ens.round(3))))
print(f"VAL macro-F1 — EffNet : {f1_single[0]:.4f} | ConvNeXt : {f1_single[1]:.4f} | "
      f"moyenne 50/50 : {f1_moy:.4f} | ensemble optimisé : {f1_ens:.4f}")

if f1_ens > f1_single[i_best] + 0.005:                  # marge contre le bruit
    W, CHOIX = W_ens, "ENSEMBLE EfficientNetV2-S + ConvNeXt-Tiny"
else:
    W = np.eye(2)[i_best]; CHOIX = f"{NAMES[i_best]} seul"
print("→ Retenu :", CHOIX, "(décidé sur VAL)")

probs_val  = ensemble_weighted(PV, W)
probs_test = ensemble_weighted(PT, W)
probs_ph2  = ensemble_weighted(PP, W)

Poids optimisés (VAL) : {'effnetv2s': np.float64(0.488), 'convnext': np.float64(0.512)}
VAL macro-F1 — EffNet : 0.8010 | ConvNeXt : 0.7964 | moyenne 50/50 : 0.8287 | ensemble optimisé : 0.8295
→ Retenu : ENSEMBLE EfficientNetV2-S + ConvNeXt-Tiny (décidé sur VAL)


## Step 10 — Temperature scaling + seuils (sensibilité mélanome ≥ 0.85), sur VAL

In [13]:
T   = find_temperature(probs_val, y_val)
thr = optimize_thresholds(apply_temperature(probs_val, T), y_val)
print(f"T = {T:.4f}")
print("Seuils :", "  ".join(f"{IDX_TO_CLASS[i]}={thr[i]:.2f}" for i in range(NUM_CLASSES)))

yv_raw = probs_val.argmax(1)
yv_cal = apply_thresholds(apply_temperature(probs_val, T), thr)
for nom, yp in [("brut", yv_raw), ("calibré", yv_cal)]:
    print(f"VAL {nom:8s} Acc={acc(y_val, yp):.4f}  F1={mf1(y_val, yp):.4f}  "
          f"SensMel={s_mel(y_val, yp):.4f}  SpecMel={sp_mel(y_val, yp):.4f}")

T = 0.6784
Seuils : akiec=1.84  bcc=2.93  bkl=2.66  df=2.09  nv=2.58  mel=0.44  vasc=0.50
VAL brut     Acc=0.8743  F1=0.8295  SensMel=0.6975  SpecMel=0.9699
VAL calibré  Acc=0.8559  F1=0.8178  SensMel=0.8532  SpecMel=0.9045


## Step 11 — TEST (une seule fois) : brut vs calibré, avec IC 95 %

In [14]:
target_names = [CLASS_NAMES_FULL[IDX_TO_CLASS[i]] for i in range(NUM_CLASSES)]
yt_raw = probs_test.argmax(1)
yt_cal = apply_thresholds(apply_temperature(probs_test, T), thr)

res_test_raw = report("TEST brut",    y_test, yt_raw)
res_test_cal = report("TEST calibré", y_test, yt_cal)

yb = label_binarize(y_test, classes=np.arange(NUM_CLASSES))
print(f"\nMacro AUC (one-vs-rest) : {roc_auc_score(yb, probs_test, average='macro'):.4f}")

for nom, yp in [("brut", yt_raw), ("calibré", yt_cal)]:
    print(f"\n--- Rapport par classe ({nom}) ---")
    print(classification_report(y_test, yp, labels=np.arange(NUM_CLASSES),
                                target_names=target_names, digits=4, zero_division=0))
print("Matrice de confusion (calibré ; lignes = vrai, colonnes = prédit) :")
print(confusion_matrix(y_test, yt_cal, labels=np.arange(NUM_CLASSES)))


=== TEST brut (3738 images) ===
  Accuracy           0.8708   IC95% [0.8606 – 0.8810]
  Macro-F1           0.8309   IC95% [0.8048 – 0.8528]
  Sensibilité mel    0.7304   IC95% [0.6964 – 0.7638]
  Spécificité mel    0.9662   IC95% [0.9597 – 0.9724]

=== TEST calibré (3738 images) ===
  Accuracy           0.8414   IC95% [0.8299 – 0.8526]
  Macro-F1           0.8104   IC95% [0.7828 – 0.8332]
  Sensibilité mel    0.8522   IC95% [0.8247 – 0.8783]
  Spécificité mel    0.8924   IC95% [0.8816 – 0.9032]

Macro AUC (one-vs-rest) : 0.9807

--- Rapport par classe (brut) ---
                               precision    recall  f1-score   support

            Actinic keratoses     0.7655    0.7255    0.7450       153
         Basal cell carcinoma     0.8793    0.9338    0.9057       468
Benign keratosis-like lesions     0.7719    0.7519    0.7618       387
               Dermatofibroma     0.8462    0.7857    0.8148        28
             Melanocytic nevi     0.9063    0.9403    0.9230      1976
   

## Step 12 — Gain par rapport à EfficientNet seul (bootstrap apparié sur le TEST)
Si la borne basse de l'IC est > 0, l'amélioration est statistiquement solide.

In [15]:
def paired_diff(yt, pred_new, pred_ref, metric, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        d.append(metric(yt[i], pred_new[i]) - metric(yt[i], pred_ref[i]))
    return metric(yt, pred_new) - metric(yt, pred_ref), np.percentile(d, [2.5, 97.5])

ref_raw = P_eff["test"].argmax(1)
print(f"Retenu ({CHOIX}) vs EfficientNetV2-S seul, TEST brut :")
gains = {}
for mname, m in [("Accuracy", acc), ("Macro-F1", mf1)]:
    d, (lo, hi) = paired_diff(y_test, yt_raw, ref_raw, m)
    gains[mname] = (float(d), float(lo), float(hi))
    verdict = "✅ significatif" if lo > 0 else ("❌ moins bon" if hi < 0 else "≈ non significatif")
    print(f"  {mname:9s} {d:+.4f}   IC95% [{lo:+.4f} ; {hi:+.4f}]   {verdict}")

Retenu (ENSEMBLE EfficientNetV2-S + ConvNeXt-Tiny) vs EfficientNetV2-S seul, TEST brut :
  Accuracy  +0.0104   IC95% [+0.0035 ; +0.0174]   ✅ significatif
  Macro-F1  +0.0414   IC95% [+0.0201 ; +0.0633]   ✅ significatif


## Step 13 — PH2 (validation externe) : brut vs calibré, avec IC 95 %

In [16]:
yt_bin = (y_ph2 == MEL).astype(int)
M_bin = {"Accuracy": acc,
         "Sensibilité mel": lambda a, b: ((a == 1) & (b == 1)).sum() / max((a == 1).sum(), 1),
         "Spécificité mel": lambda a, b: ((a == 0) & (b == 0)).sum() / max((a == 0).sum(), 1)}

yp_raw = (probs_ph2.argmax(1) == MEL).astype(int)
yp_cal = (apply_thresholds(apply_temperature(probs_ph2, T), thr) == MEL).astype(int)
res_ph2_raw = report("PH2 brut",    yt_bin, yp_raw, M_bin)
res_ph2_cal = report("PH2 calibré", yt_bin, yp_cal, M_bin)
auc_ph2 = roc_auc_score(yt_bin, probs_ph2[:, MEL])
print(f"\nAUC mélanome PH2 : {auc_ph2:.4f}")
print("Matrice (calibré ; lignes = vrai [non-mel, mel]) :")
print(confusion_matrix(yt_bin, yp_cal))


=== PH2 brut (197 images) ===
  Accuracy           0.8376   IC95% [0.7817 – 0.8883]
  Sensibilité mel    0.4808   IC95% [0.3396 – 0.6200]
  Spécificité mel    0.9655   IC95% [0.9333 – 0.9931]

=== PH2 calibré (197 images) ===
  Accuracy           0.8782   IC95% [0.8325 – 0.9239]
  Sensibilité mel    0.7308   IC95% [0.6042 – 0.8491]
  Spécificité mel    0.9310   IC95% [0.8865 – 0.9720]

AUC mélanome PH2 : 0.8479
Matrice (calibré ; lignes = vrai [non-mel, mel]) :
[[135  10]
 [ 14  38]]


## Step 14 — Sauvegarde des artefacts

In [17]:
import json

np.save(f"{OUT_DIR}/ensemble_weights.npy", W)
np.save(f"{OUT_DIR}/temperature.npy", np.array([T]))
np.save(f"{OUT_DIR}/thresholds.npy", thr)
np.save(f"{OUT_DIR}/probs_val.npy",  probs_val)
np.save(f"{OUT_DIR}/probs_test.npy", probs_test)
np.save(f"{OUT_DIR}/probs_ph2.npy",  probs_ph2)

for nom, df in [("train", train_df), ("val", val_df), ("test", test_df), ("ph2", ph2_df)]:
    df.to_csv(f"{OUT_DIR}/split_{nom}.csv", index=False)

pd.DataFrame({"image_uid": test_df["image_uid"].values,
              "true": [IDX_TO_CLASS[i] for i in y_test],
              "pred_raw": [IDX_TO_CLASS[i] for i in yt_raw],
              "pred_cal": [IDX_TO_CLASS[i] for i in yt_cal],
              "confidence": apply_temperature(probs_test, T).max(1)}
            ).to_csv(f"{OUT_DIR}/predictions_test.csv", index=False)

config = {
    "modeles": {"effnetv2s": "EfficientNetV2-S + CBAM multi-échelle (notebook skincanerf-phase3)",
                "convnext": "ConvNeXt-Tiny, GAP + LayerNorm + Dropout 0.3"},
    "loss": "categorical crossentropy + label smoothing 0.1 (pas de focal loss)",
    "convnext_entrainement": {"phase1_epochs": int(ep_p1), "phase1_best_f1": float(best_p1),
                              "phase2_epochs": int(ep_p2), "phase2_best_f1": float(best_p2),
                              "duree_h": round((time.time() - T0) / 3600, 2)},
    "noms": NAMES, "poids": W.tolist(), "choix": CHOIX, "gain_vs_effnet_test": gains,
    "temperature": float(T), "seuils": thr.tolist(), "cible_sens_mel": CIBLE_SENS_MEL,
    "test_brut": res_test_raw, "test_calibre": res_test_cal,
    "ph2_brut": res_ph2_raw, "ph2_calibre": res_ph2_cal, "ph2_auc_mel": float(auc_ph2),
}
with open(f"{OUT_DIR}/run_config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:40s} {os.path.getsize(f'{OUT_DIR}/{f}') / 1e6:8.2f} MB")

  __notebook__.ipynb                           6.15 MB
  convnext_tiny_final.keras                  557.34 MB
  convnext_tiny_phase1.keras                 111.95 MB
  ensemble_weights.npy                         0.00 MB
  predictions_test.csv                         0.16 MB
  preds_ph2_convnext.npy                       0.01 MB
  preds_test_convnext.npy                      0.21 MB
  preds_val_convnext.npy                       0.21 MB
  probs_ph2.npy                                0.01 MB
  probs_test.npy                               0.21 MB
  probs_val.npy                                0.21 MB
  run_config.json                              0.00 MB
  split_ph2.csv                                0.03 MB
  split_test.csv                               0.46 MB
  split_train.csv                              2.15 MB
  split_val.csv                                0.46 MB
  temperature.npy                              0.00 MB
  thresholds.npy                               0.00 MB


## Step 15 — Compromis sensibilité / spécificité (seuils choisis sur VAL, appliqués au TEST)

In [18]:
probs_val_T = apply_temperature(probs_val, T)
print("Cible sens. mel | Acc test | Macro-F1 | Sens. mel test | Spéc. mel test")
for cible in [0.70, 0.75, 0.80, 0.85, 0.90]:
    CIBLE_SENS_MEL = cible
    t  = optimize_thresholds(probs_val_T, y_val)
    yp = apply_thresholds(apply_temperature(probs_test, T), t)
    print(f"     {cible:.2f}      |  {acc(y_test, yp):.4f}  |  {mf1(y_test, yp):.4f}  |"
          f"     {s_mel(y_test, yp):.4f}     |    {sp_mel(y_test, yp):.4f}")
CIBLE_SENS_MEL = 0.85

Cible sens. mel | Acc test | Macro-F1 | Sens. mel test | Spéc. mel test
     0.70      |  0.8703  |  0.8257  |     0.7739     |    0.9534
     0.75      |  0.8657  |  0.8201  |     0.8000     |    0.9426
     0.80      |  0.8636  |  0.8265  |     0.8101     |    0.9357
     0.85      |  0.8414  |  0.8104  |     0.8522     |    0.8924
     0.90      |  0.8127  |  0.7878  |     0.8928     |    0.8468
